In [41]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

In [42]:
# load back in the RNA fingerprints and split prepared in Task3_01
DATA_DIR = "/home/ubuntu/data/frangieh"

pert_FC_selected = pd.read_pickle(f"{DATA_DIR}/task3_pert_FC_selected_50.pkl")
train_40 = pd.read_csv(f"{DATA_DIR}/task3_train_40.csv")["perturbation"].tolist()
test_10 = pd.read_csv(f"{DATA_DIR}/task3_test_10.csv")["perturbation"].tolist()
gene_cols = pert_FC_selected.columns.tolist()
selected_50 = train_40 + test_10
conditions = pert_FC_selected.index.get_level_values("condition").unique().tolist()

pert_FC_selected.shape

(150, 2042)

In [44]:

# ---------------------------------------------------------------
# 2. Gene "identity" features (leakage-free, generalizes to unseen genes)
# ---------------------------------------------------------------
def build_gene_embeddings(fc_df, train_perts, gene_cols, n_components=32):
    """
    Embed every gene in `gene_cols` using its own column of log2FC values,
    restricted to rows whose perturbation is in `train_perts`.
    Shape going in: (n_train_rows, n_genes) -> transpose -> genes as samples.
    """
    train_rows = fc_df.index.get_level_values("perturbation").isin(train_perts)
    train_block = fc_df.loc[train_rows]  # (n_train_rows, n_genes)
 
    gene_signatures = train_block.T.values  # (n_genes, n_train_rows)
 
    scaler = StandardScaler()
    gene_signatures = scaler.fit_transform(gene_signatures)
 
    n_components = min(n_components, gene_signatures.shape[0], gene_signatures.shape[1])
    pca = PCA(n_components=n_components, random_state=SEED)
    embeddings = pca.fit_transform(gene_signatures)  # (n_genes, n_components)
 
    emb_df = pd.DataFrame(embeddings, index=gene_cols)
    return emb_df, pca, scaler
 
 
def get_gene_embedding(gene, emb_df, dim):
    if gene in emb_df.index:
        return emb_df.loc[gene].values.astype(np.float32)
    # gene wasn't itself profiled as a measured column -> fall back to zeros
    return np.zeros(dim, dtype=np.float32)
 
 
# ---------------------------------------------------------------
# 3. Split off validation perturbations FIRST, before anything is fit.
#    This must happen before gene_embeddings / target_pca so those can't
#    see val_perts -- mirrors exactly how test_10 is treated.
# ---------------------------------------------------------------
val_perts = list(np.random.choice(train_40, size=max(1, len(train_40) // 5), replace=False))
fit_perts = [p for p in train_40 if p not in val_perts]
 
# ---------------------------------------------------------------
# 4. Gene embeddings -- fit on fit_perts only, not all of train_40
# ---------------------------------------------------------------
gene_embeddings, gene_pca, gene_scaler = build_gene_embeddings(
    pert_FC_selected, fit_perts, gene_cols, n_components=32
)
EMB_DIM = gene_embeddings.shape[1]
 
# ---------------------------------------------------------------
# 5. Target compression (PCA fit on fit_perts only)
# ---------------------------------------------------------------
fit_mask = pert_FC_selected.index.get_level_values("perturbation").isin(fit_perts)
Y_fit_full = pert_FC_selected.loc[fit_mask].values
 
N_TARGET_PCS = min(35, Y_fit_full.shape[0] - 1)  # keep well below n_fit_rows
target_pca = PCA(n_components=N_TARGET_PCS, random_state=SEED)
target_pca.fit(Y_fit_full)
print(f"Target PCA explained variance ({N_TARGET_PCS} PCs): "
      f"{target_pca.explained_variance_ratio_.sum():.3f}")
 
# ---------------------------------------------------------------
# 6. Build (X, Y) arrays, one row per (perturbation, condition)
# ---------------------------------------------------------------
cond_to_idx = {c: i for i, c in enumerate(conditions)}
 
def build_dataset(perturbations, fc_df, emb_df, target_pca_model):
    X, Y_pc, Y_full, meta = [], [], [], []
    for pert in perturbations:
        sub = fc_df.loc[pert]  # index = condition
        for cond in sub.index.get_level_values("condition"):
            fc_vec = sub.loc[cond].values.astype(np.float32)
            gene_emb = get_gene_embedding(pert, emb_df, EMB_DIM)
            cond_oh = np.zeros(len(conditions), dtype=np.float32)
            cond_oh[cond_to_idx[cond]] = 1.0
            X.append(np.concatenate([gene_emb, cond_oh]))
            Y_pc.append(target_pca_model.transform(fc_vec[None, :])[0])
            Y_full.append(fc_vec)
            meta.append((pert, cond))
    return (np.stack(X).astype(np.float32),
            np.stack(Y_pc).astype(np.float32),
            np.stack(Y_full).astype(np.float32),
            meta)
 
X_fit, Y_fit_pc, _, meta_fit = build_dataset(fit_perts, pert_FC_selected, gene_embeddings, target_pca)
X_val, Y_val_pc, Y_val_true, meta_val = build_dataset(val_perts, pert_FC_selected, gene_embeddings, target_pca)
X_test, Y_test_pc, Y_test_true, meta_test = build_dataset(test_10, pert_FC_selected, gene_embeddings, target_pca)
 
# ---------------------------------------------------------------
# 7. Dataset / DataLoader
# ---------------------------------------------------------------
class PertDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.from_numpy(X)
        self.Y = torch.from_numpy(Y)
 
    def __len__(self):
        return len(self.X)
 
    def __getitem__(self, i):
        return self.X[i], self.Y[i]
 
 
train_loader = DataLoader(PertDataset(X_fit, Y_fit_pc), batch_size=16, shuffle=True)
 
# ---------------------------------------------------------------
# 8. Model
# ---------------------------------------------------------------
class PerturbationMLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim),
        )
 
    def forward(self, x):
        return self.net(x)
 
 
model = PerturbationMLP(in_dim=X_fit.shape[1], out_dim=N_TARGET_PCS)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.MSELoss()
 
# ---------------------------------------------------------------
# 9. Train (early-stopping on held-out-perturbation validation set)
# ---------------------------------------------------------------
N_EPOCHS = 200
best_val = np.inf
best_state = None
 
X_val_t = torch.from_numpy(X_val)
Y_val_pc_t = torch.from_numpy(Y_val_pc)
 
for epoch in range(N_EPOCHS):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        optimizer.step()
 
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t)
        val_loss = loss_fn(val_pred, Y_val_pc_t).item()
    if val_loss < best_val:
        best_val = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    if epoch % 20 == 0:
        print(f"epoch {epoch:3d}  val_mse={val_loss:.4f}")
 
model.load_state_dict(best_state)
 
# ---------------------------------------------------------------
# 10. Baseline: per-condition mean of training perturbations
# ---------------------------------------------------------------
def condition_mean_baseline(fc_df, train_perts):
    train_rows = fc_df.loc[fc_df.index.get_level_values("perturbation").isin(train_perts)]
    return train_rows.groupby(level="condition").mean()  # (n_conditions, n_genes)
 
baseline_means = condition_mean_baseline(pert_FC_selected, train_40)
 
# ---------------------------------------------------------------
# 11. Evaluate on held-out test_10 perturbations
# ---------------------------------------------------------------
def evaluate(model, target_pca_model, X, Y_true_full, meta, baseline_means):
    model.eval()
    with torch.no_grad():
        pred_pc = model(torch.from_numpy(X)).numpy()
    pred_full = target_pca_model.inverse_transform(pred_pc)
 
    rows = []
    for i, (pert, cond) in enumerate(meta):
        y_true = Y_true_full[i]
        y_pred = pred_full[i]
        y_base = baseline_means.loc[cond].values
 
        r_model, _ = pearsonr(y_true, y_pred)
        r_base, _ = pearsonr(y_true, y_base)
        mse_model = np.mean((y_true - y_pred) ** 2)
        mse_base = np.mean((y_true - y_base) ** 2)
        spearman_r_model, _ = spearmanr(y_true, y_pred)
        spearman_r_base, _ = spearmanr(y_true, y_base)
        rows.append(dict(perturbation=pert, condition=cond,
                          pearson_model=r_model, pearson_baseline=r_base,
                          mse_model=mse_model, mse_baseline=mse_base,
                          spearman_base=spearman_r_base, spearman_model=spearman_r_model))
    return pd.DataFrame(rows)
 
results_df = evaluate(model, target_pca, X_test, Y_test_true, meta_test, baseline_means)
print(results_df.groupby("condition")[["pearson_model",
                                        "mse_model", "spearman_model"]].mean())



Target PCA explained variance (35 PCs): 0.841
epoch   0  val_mse=0.0795
epoch  20  val_mse=0.0749
epoch  40  val_mse=0.0736
epoch  60  val_mse=0.0768
epoch  80  val_mse=0.0805
epoch 100  val_mse=0.0913
epoch 120  val_mse=0.0905
epoch 140  val_mse=0.0983
epoch 160  val_mse=0.1032
epoch 180  val_mse=0.1067
            pearson_model  mse_model  spearman_model
condition                                           
Co-culture       0.594466   0.003699        0.242133
Control          0.753014   0.002486        0.358463
IFNγ             0.716824   0.002369        0.338408
